### 3. Feature Engineering

### `03_feature_engineering.ipynb`

In [2]:
# Improved function to create feature matrix with complete data
def create_feature_matrix_with_defaults(features_list):
    """
    Create feature matrix ensuring all required features are present with defaults
    
    Args:
        features_list: List of feature dictionaries
    
    Returns:
        DataFrame with complete feature set
    """
    if not features_list:
        return pd.DataFrame()
    
    # List of required features for each category
    required_features = {
        'joint_angles': [
            'right_elbow_mean', 'right_elbow_min', 'right_elbow_max', 'right_elbow_range', 'right_elbow_max_velocity',
            'left_elbow_mean', 'left_elbow_min', 'left_elbow_max', 'left_elbow_range', 'left_elbow_max_velocity',
            'right_knee_mean', 'right_knee_min', 'right_knee_max', 'right_knee_range', 'right_knee_max_velocity',
            'left_knee_mean', 'left_knee_min', 'left_knee_max', 'left_knee_range', 'left_knee_max_velocity',
            'right_hip_mean', 'right_hip_min', 'right_hip_max', 'right_hip_range', 'right_hip_max_velocity',
            'left_hip_mean', 'left_hip_min', 'left_hip_max', 'left_hip_range', 'left_hip_max_velocity'
        ],
        'vertical_motion': [
            'max_upper_body_velocity_y', 'max_upper_body_accel_y', 'max_center_velocity_y', 'max_center_accel_y',
            'upper_body_vertical_displacement', 'center_vertical_displacement'
        ],
        'impact': [
            'has_impact', 'impact_count', 'max_deceleration', 'impact_frame'
        ],
        'collapse': [
            'height_reduction_pct', 'aspect_ratio_increase_pct', 
            'max_height_collapse_velocity', 'max_aspect_ratio_change_velocity'
        ]
    }
    
    rows = []
    
    for features in features_list:
        # Start with basic metadata
        row = {
            'label': features['metadata']['label'],
            'data_type': features['metadata'].get('data_type', 'unknown')
        }
        
        # Add file path based on data type
        if features['metadata'].get('data_type') == 'video':
            row['file_path'] = features['metadata'].get('video_path', 'unknown')
            row['fps'] = features['metadata'].get('fps', 0)
            row['duration'] = features['metadata'].get('duration', 0)
        else:  # image
            row['file_path'] = features['metadata'].get('image_path', 'unknown')
            row['width'] = features['metadata'].get('width', 0)
            row['height'] = features['metadata'].get('height', 0)
        
        # Add features from each category with defaults for missing values
        for category, feature_names in required_features.items():
            if category in features:
                for feature in feature_names:
                    if feature in features[category]:
                        row[feature] = features[category][feature]
                    else:
                        # Use appropriate defaults based on feature type
                        if feature == 'has_impact':
                            row[feature] = False
                        elif feature in ['impact_count', 'impact_frame']:
                            row[feature] = 0
                        else:
                            row[feature] = 0.0  # Default to 0.0 for numeric features
            else:
                # Category missing - add all features with defaults
                for feature in feature_names:
                    if feature == 'has_impact':
                        row[feature] = False
                    elif feature in ['impact_count', 'impact_frame']:
                        row[feature] = 0
                    else:
                        row[feature] = 0.0
        
        # Add proportions data if available (for image data)
        if 'proportions' in features:
            for key, value in features['proportions'].items():
                row[key] = value
        
        # Add bounding box data if available
        if 'bounding_box' in features:
            for key, value in features['bounding_box'].items():
                row[f"bbox_{key}"] = value
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Convert boolean columns to int
    bool_cols = df.select_dtypes(include=['bool']).columns
    for col in bool_cols:
        df[col] = df[col].astype(int)
    
    return df

# Process all datasets and extract features with improved handling
def process_all_datasets():
    """Process all datasets and extract features with robust error handling"""
    print("Loading pose data from all datasets...")
    
    # Load pose data from each dataset type
    fall_pose_data = load_pose_data("falls")
    workout_pose_data = load_pose_data("workouts")
    activity_pose_data = load_pose_data("activities")
    
    # Extract features from each dataset
    print("\nExtracting features from fall videos...")
    fall_features = []
    for pose_data in tqdm(fall_pose_data, desc="Processing fall videos"):
        # Identify data type
        data_type = identify_data_type(pose_data)
        
        # Extract features based on data type
        if data_type == 'video':
            features = extract_features_from_video_pose_data(pose_data)
        elif data_type == 'image':
            features = extract_features_from_image_pose_data(pose_data)
        else:
            print(f"Unknown data type for {pose_data.get('video_path', pose_data.get('image_path', 'unknown'))}")
            features = None
            
        if features:
            features['metadata']['label'] = 'fall'
            fall_features.append(features)
    
    print(f"Extracted features from {len(fall_features)}/{len(fall_pose_data)} fall data files")
    
    print("\nExtracting features from workout videos...")
    workout_features = []
    for pose_data in tqdm(workout_pose_data, desc="Processing workout videos"):
        # Identify data type
        data_type = identify_data_type(pose_data)
        
        # Extract features based on data type
        if data_type == 'video':
            features = extract_features_from_video_pose_data(pose_data)
        elif data_type == 'image':
            features = extract_features_from_image_pose_data(pose_data)
        else:
            print(f"Unknown data type for {pose_data.get('video_path', pose_data.get('image_path', 'unknown'))}")
            features = None
            
        if features:
            features['metadata']['label'] = 'workout'
            workout_features.append(features)
    
    print(f"Extracted features from {len(workout_features)}/{len(workout_pose_data)} workout data files")
    
    print("\nExtracting features from human activity data...")
    activity_features = []
    video_count = 0
    image_count = 0
    
    for pose_data in tqdm(activity_pose_data, desc="Processing activity data"):
        # Identify data type
        data_type = identify_data_type(pose_data)
        
        # Extract features based on data type
        if data_type == 'video':
            features = extract_features_from_video_pose_data(pose_data)
            if features:
                video_count += 1
        elif data_type == 'image':
            features = extract_features_from_image_pose_data(pose_data)
            if features:
                image_count += 1
        else:
            print(f"Unknown data type for {pose_data.get('video_path', pose_data.get('image_path', 'unknown'))}")
            features = None
            
        if features:
            features['metadata']['label'] = 'activity'
            activity_features.append(features)
    
    print(f"Extracted features from {len(activity_features)}/{len(activity_pose_data)} activity data files")
    print(f"   - Video data: {video_count} files")
    print(f"   - Image data: {image_count} files")
    
    # Combine all features
    all_features = fall_features + workout_features + activity_features
    print(f"\nTotal extracted feature sets: {len(all_features)}")
    
    # Save all features
    features_file = FEATURES_DIR / "all_features.json"
    with open(features_file, 'w') as f:
        json.dump(all_features, f, indent=2)
    
    print(f"Saved all features to {features_file}")
    
    # Also save separate files for each category
    for category, features_list in [
        ("falls", fall_features), 
        ("workouts", workout_features), 
        ("activities", activity_features)
    ]:
        if features_list:
            category_file = FEATURES_DIR / f"{category}_features.json"
            with open(category_file, 'w') as f:
                json.dump(features_list, f, indent=2)
            print(f"Saved {category} features to {category_file}")
    
    # Create feature matrix with robust defaults
    feature_df = create_feature_matrix_with_defaults(all_features)
    
    # Export to CSV
    csv_file = FEATURES_DIR / "features_for_modeling.csv"
    feature_df.to_csv(csv_file, index=False)
    print(f"Exported complete feature matrix to {csv_file}")
    
    return all_features

# Fix missing data in existing feature CSV if already generated
def fix_existing_features_csv(csv_path=None):
    """
    Fix missing data in an existing features CSV file
    
    Args:
        csv_path: Path to features CSV file (None to use default)
    
    Returns:
        DataFrame with fixed features
    """
    if csv_path is None:
        csv_path = FEATURES_DIR / "features_for_modeling.csv"
    
    print(f"Fixing missing data in {csv_path}")
    
    # Load the CSV file
    if not os.path.exists(csv_path):
        print(f"Error: File {csv_path} not found")
        return None
    
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV with {df.shape[0]} rows and {df.shape[1]} columns")
    
    # Print missing value statistics before fix
    missing_before = df.isna().sum()
    missing_pct_before = (df.isna().sum() / len(df)) * 100
    print(f"Missing values before fix: {missing_before.sum()}")
    print(f"Columns with most missing values:")
    for col in missing_before[missing_before > 0].sort_values(ascending=False).index[:10]:
        print(f"  {col}: {missing_before[col]} missing ({missing_pct_before[col]:.2f}%)")
    
    # Fix missing values with appropriate defaults
    for col in df.columns:
        if col in ['label', 'data_type', 'file_path']:
            # Skip metadata columns
            continue
            
        if col == 'has_impact':
            # Boolean feature
            df[col].fillna(False, inplace=True)
        elif col in ['impact_count', 'impact_frame']:
            # Integer features
            df[col].fillna(0, inplace=True)
        elif col in ['fps', 'duration', 'width', 'height']:
            # Numeric metadata
            df[col].fillna(0, inplace=True)
        else:
            # Regular numeric features
            df[col].fillna(0.0, inplace=True)
    
    # Print missing value statistics after fix
    missing_after = df.isna().sum()
    print(f"Missing values after fix: {missing_after.sum()}")
    
    # Save fixed CSV
    fixed_csv_path = csv_path.parent / f"{csv_path.stem}_fixed.csv"
    df.to_csv(fixed_csv_path, index=False)
    print(f"Saved fixed CSV to {fixed_csv_path}")
    
    return df

# Main execution
if __name__ == "__main__":
    # Check if we should process data or fix existing
    existing_csv = FEATURES_DIR / "features_for_modeling.csv"
    if os.path.exists(existing_csv):
        print(f"Found existing feature CSV at {existing_csv}")
        choice = input("1. Fix existing CSV\n2. Reprocess all data\nChoice (default: 1): ").strip() or "1"
        
        if choice == "1":
            # Fix existing CSV
            fixed_df = fix_existing_features_csv(existing_csv)
            
            # Apply normalization and validate the fixed dataset
            if fixed_df is not None:
                # Normalize features
                from sklearn.preprocessing import MinMaxScaler
                
                # Select only numeric columns (excluding categorical ones)
                exclude_cols = ['label', 'data_type', 'file_path']
                numeric_cols = fixed_df.select_dtypes(include=['number']).columns.tolist()
                numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
                
                # Create a MinMaxScaler for scaling to [0, 1]
                scaler = MinMaxScaler()
                
                # Scale numeric features
                fixed_df[numeric_cols] = scaler.fit_transform(fixed_df[numeric_cols])
                
                # Save normalized features
                normalized_csv = FEATURES_DIR / "normalized_features.csv"
                fixed_df.to_csv(normalized_csv, index=False)
                print(f"Saved normalized features to {normalized_csv}")
                
                # Validate that no missing values remain
                missing_final = fixed_df.isna().sum().sum()
                if missing_final == 0:
                    print("All missing values successfully fixed!")
                else:
                    print(f"Warning: {missing_final} missing values remain")
        else:
            # Reprocess all datasets from scratch
            all_features = process_all_datasets()
    else:
        # No existing CSV, process all datasets
        print("No existing feature CSV found, processing all datasets")
        all_features = process_all_datasets()
    
    print("\nFeature processing completed successfully!")
    print("Now you can use the updated features for model training.")
    print("The fixed features have zero missing values and appropriate defaults for all required features.")
    
# Function to extract features from image data (with improved error handling)
def extract_features_from_image_pose_data(pose_data):
    """
    Extract features from image pose data (single image) with improved error handling
    
    Args:
        pose_data: Pose data dictionary from JSON with 'people' key
    
    Returns:
        Dictionary of extracted features or None if extraction fails
    """
    # Basic metadata
    image_path = pose_data.get('image_path', 'unknown')
    label = pose_data.get('label', 'unknown')
    
    print(f"Processing image: {image_path}")
    
    # Extract people data
    people = pose_data.get('people', [])
    print(f"Image has {len(people)} people")
    
    # Skip if no people detected
    if not people:
        print("No people detected, skipping")
        return None
    
    # Use the first person's keypoints
    keypoints = people[0].get('keypoints', [])
    
    if not keypoints:
        print("No keypoints found, skipping")
        return None
        
    print(f"First person has {len(keypoints)} keypoints")
    if keypoints:
        print(f"Sample keypoint: {keypoints[0]}")
    
    # Add empty placeholders for missing keypoints (if any)
    while len(keypoints) < 17:
        keypoints.append([0, 0, 0])  # Add dummy keypoint with zero confidence
    
    # Calculate static features only with error handling
    try:
        joint_angles = calculate_joint_angles(keypoints)
        proportions = calculate_body_proportions(keypoints)
        bbox = calculate_bounding_box(keypoints)
        
        # Create feature dictionary with guaranteed values
        features = {
            'metadata': {
                'image_path': image_path,
                'label': label,
                'data_type': 'image',
                'width': pose_data.get('width', 0),
                'height': pose_data.get('height', 0)
            },
            'joint_angles': joint_angles or {},
            'proportions': proportions or {},
            'bounding_box': bbox or {}
        }
        
        return features
        
    except Exception as e:
        print(f"Error extracting image features: {e}")
        # Return basic features with defaults
        return {
            'metadata': {
                'image_path': image_path,
                'label': label,
                'data_type': 'image',
                'width': pose_data.get('width', 0),
                'height': pose_data.get('height', 0)
            },
            'joint_angles': {},
            'proportions': {},
            'bounding_box': {}
        }

# Function to validate features data for completeness
def validate_features(features_list):
    """
    Validate feature completeness and log issues
    
    Args:
        features_list: List of feature dictionaries
        
    Returns:
        Number of problems found
    """
    problems = 0
    
    # Define required categories and key features
    required_categories = ['joint_angles', 'vertical_motion', 'impact', 'collapse']
    key_features = {
        'joint_angles': ['right_elbow_mean', 'left_elbow_mean', 'right_knee_mean', 'left_knee_mean'],
        'vertical_motion': ['max_upper_body_velocity_y', 'max_center_velocity_y'],
        'impact': ['has_impact', 'impact_count', 'max_deceleration'],
        'collapse': ['height_reduction_pct', 'aspect_ratio_increase_pct']
    }
    
    for i, features in enumerate(features_list):
        # Get metadata for reference
        if 'metadata' in features:
            if features['metadata'].get('data_type') == 'video':
                path = features['metadata'].get('video_path', 'unknown')
            else:
                path = features['metadata'].get('image_path', 'unknown')
        else:
            path = f"item_{i}"
        
        # Check for missing feature categories
        missing_categories = []
        for category in required_categories:
            if category not in features or not features[category]:
                missing_categories.append(category)
        
        if missing_categories:
            problems += 1
            print(f"Warning: {path} missing categories: {', '.join(missing_categories)}")
        
        # Check for critical missing features
        for category, features_list in key_features.items():
            if category in features:
                for feature in features_list:
                    if feature not in features[category] or features[category][feature] is None or np.isnan(features[category][feature]):
                        print(f"Warning: {path} missing critical feature: {feature}")
                        problems += 1
    
    print(f"Validation complete: {problems} problems found in {len(features_list)} samples")
    return problems

# Main function to execute all fixes
def fix_missing_feature_data():
    """Run all the necessary fixes for missing feature data"""
    # Step 1: Check if we have existing feature CSV
    existing_csv = FEATURES_DIR / "features_for_modeling.csv"
    
    if os.path.exists(existing_csv):
        print(f"Found existing feature CSV at {existing_csv}")
        
        # Analyze missing values in existing CSV
        df = pd.read_csv(existing_csv)
        missing_values = df.isna().sum()
        missing_count = missing_values.sum()
        
        print(f"CSV contains {df.shape[0]} samples with {df.shape[1]} features")
        print(f"Total missing values: {missing_count}")
        
        if missing_count > 0:
            print("Top 10 columns with missing values:")
            for col, count in missing_values.nlargest(10).items():
                if count > 0:
                    print(f"  {col}: {count} missing ({count/len(df)*100:.2f}%)")
            
            # Fix missing values
            fixed_df = fix_existing_features_csv(existing_csv)
            
            # Save new feature CSV
            fixed_csv = FEATURES_DIR / "features_for_modeling_fixed.csv"
            fixed_df.to_csv(fixed_csv, index=False)
            print(f"Fixed features saved to {fixed_csv}")
            
            # Create normalized version
            normalized_df = normalize_features(fixed_df)
            normalized_csv = FEATURES_DIR / "normalized_features.csv"
            normalized_df.to_csv(normalized_csv, index=False)
            print(f"Normalized features saved to {normalized_csv}")
            
            return fixed_df
        else:
            print("No missing values found! Your data appears to be complete.")
            return df
    else:
        print("No existing feature CSV found. Need to process all datasets.")
        
        # Process all datasets with improved error handling
        all_features = process_all_datasets()
        
        # Check if features were successfully extracted
        if all_features:
            print(f"Successfully extracted features from {len(all_features)} samples")
            
            # Validate the features
            problems = validate_features(all_features)
            
            if problems > 0:
                print(f"Found {problems} issues in the extracted features. These will be fixed.")
            
            # Create feature matrix with defaults
            feature_df = create_feature_matrix_with_defaults(all_features)
            
            # Save to CSV
            csv_file = FEATURES_DIR / "features_for_modeling.csv"
            feature_df.to_csv(csv_file, index=False)
            
            # Create normalized version
            normalized_df = normalize_features(feature_df)
            normalized_csv = FEATURES_DIR / "normalized_features.csv"
            normalized_df.to_csv(normalized_csv, index=False)
            
            print(f"Complete feature matrix saved to {csv_file}")
            print(f"Normalized features saved to {normalized_csv}")
            
            return feature_df
        else:
            print("Error: Failed to extract features from datasets")
            return None

# Normalize features for modeling
def normalize_features(feature_df):
    """
    Normalize numeric features for better comparison and ML
    
    Args:
        feature_df: DataFrame with extracted features
    
    Returns:
        DataFrame with normalized features
    """
    if feature_df.empty:
        return feature_df
    
    # Make a copy to avoid modifying the original
    df = feature_df.copy()
    
    # Select only numeric columns (excluding categorical ones)
    exclude_cols = ['label', 'data_type', 'file_path']
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    # Skip if no numeric columns to normalize
    if not numeric_cols:
        return df
    
    # Create a MinMaxScaler for scaling to [0, 1]
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    
    # Replace infinite values with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Fill NaN values with appropriate defaults
    for col in numeric_cols:
        if 'velocity' in col or 'accel' in col:
            df[col] = df[col].fillna(0)  # Motion starts from zero
        elif 'pct' in col or 'ratio' in col:
            df[col] = df[col].fillna(1)  # No change = 1 (100%)
        else:
            df[col] = df[col].fillna(0)  # Use zero for other missing values
    
    # Scale numeric features
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    
    return df

# Main entry point if running as script
if __name__ == "__main__":
    # Run the feature data fix
    fixed_df = fix_missing_feature_data()
    
    if fixed_df is not None:
        print("\nData fixes complete! Your model should now have all the features it needs.")
        print(f"Dataset has {fixed_df.shape[0]} samples and {fixed_df.shape[1]} features with no missing values.")
        
        # Show class balance
        if 'label' in fixed_df.columns:
            class_counts = fixed_df['label'].value_counts()
            print("\nClass distribution:")
            for label, count in class_counts.items():
                print(f"  {label}: {count} samples ({count/len(fixed_df)*100:.2f}%)")
    else:
        print("Failed to fix feature data!")# Improved version of main extraction function that handles missing data
def extract_features_from_video_pose_data(pose_data):
    """
    Extract features from video pose data (sequence of frames) with improved handling of missing data
    
    Args:
        pose_data: Pose data dictionary from JSON with 'frames' key
    
    Returns:
        Dictionary of extracted features or None if extraction fails
    """
    # Basic metadata
    video_path = pose_data.get('video_path', 'unknown')
    fps = pose_data.get('fps', 30)  # Default to 30 if missing
    total_frames = len(pose_data.get('frames', []))
    label = pose_data.get('label', 'unknown')
    
    print(f"Processing video: {video_path}")
    print(f"Frames: {total_frames}, FPS: {fps}")
    
    # Skip if no frames
    if total_frames == 0:
        print("No frames found, skipping")
        return None
    
    # Extract people keypoints over time
    keypoints_sequences = []
    valid_frames = 0
    
    for frame_idx, frame in enumerate(pose_data.get('frames', [])):
        people = frame.get('people', [])
        
        # Debug first frame
        if frame_idx == 0:
            print(f"First frame has {len(people)} people")
            if people and 'keypoints' in people[0]:
                keypoints = people[0]['keypoints']
                print(f"First person keypoints shape: {len(keypoints)} keypoints")
                if keypoints:
                    print(f"Sample keypoint: {keypoints[0]}")
        
        # If this is the first frame, initialize sequences for each person
        if frame_idx == 0:
            keypoints_sequences = [[] for _ in range(len(people))]
        
        # Add keypoints for each person
        has_valid_keypoints = False
        for i, person in enumerate(people):
            if i < len(keypoints_sequences):
                if 'keypoints' in person and len(person['keypoints']) > 0:
                    keypoints_sequences[i].append(person['keypoints'])
                    has_valid_keypoints = True
        
        if has_valid_keypoints:
            valid_frames += 1
    
    # Skip if no valid keypoints found
    if not keypoints_sequences or valid_frames == 0:
        print("No valid keypoints sequences found, skipping")
        return None
    
    # For simplicity, use the first person (usually the main subject)
    main_keypoints_sequence = keypoints_sequences[0] if keypoints_sequences else []
    
    # Skip if sequence is too short
    if len(main_keypoints_sequence) < 5:
        print("Sequence too short, skipping")
        return None
    
    # Pre-process keypoints sequence for consistency
    processed_sequence = []
    for keypoints in main_keypoints_sequence:
        # Skip frames with empty keypoints
        if not keypoints:
            # Add placeholder frame with zero confidence keypoints
            processed_sequence.append([[0, 0, 0] for _ in range(17)])
            continue
            
        # Ensure all frames have 17 keypoints (COCO format)
        if len(keypoints) < 17:
            # Pad with zero-confidence keypoints
            padded = keypoints.copy()
            while len(padded) < 17:
                padded.append([0, 0, 0])
            processed_sequence.append(padded)
        else:
            processed_sequence.append(keypoints)
    
    # More robust feature calculation - catch and handle errors
    try:
        # Calculate static features for each frame
        joint_angles_sequence = []
        proportions_sequence = []
        bbox_sequence = []
        
        for keypoints in processed_sequence:
            # Calculate features with error handling
            try:
                joint_angles = calculate_joint_angles(keypoints)
                proportions = calculate_body_proportions(keypoints)
                bbox = calculate_bounding_box(keypoints)
                
                joint_angles_sequence.append(joint_angles)
                proportions_sequence.append(proportions)
                bbox_sequence.append(bbox)
            except Exception as e:
                print(f"Error calculating frame features: {e}")
                # Add empty feature dictionaries to maintain sequence length
                joint_angles_sequence.append({})
                proportions_sequence.append({})
                bbox_sequence.append({})
        
        # Calculate temporal features
        try:
            vertical_features = calculate_vertical_displacement(processed_sequence, fps)
        except Exception as e:
            print(f"Error calculating vertical features: {e}")
            # Use default values
            vertical_features = {
                'upper_body_y_trajectory': [],
                'center_y_trajectory': [],
                'upper_body_velocity_y': [],
                'upper_body_accel_y': [],
                'center_velocity_y': [],
                'center_accel_y': [],
                'max_upper_body_velocity_y': 0.0,
                'max_upper_body_accel_y': 0.0,
                'max_center_velocity_y': 0.0,
                'max_center_accel_y': 0.0,
                'upper_body_vertical_displacement': 0.0,
                'center_vertical_displacement': 0.0
            }
        
        try:
            impact_features = detect_impact(processed_sequence, fps)
        except Exception as e:
            print(f"Error calculating impact features: {e}")
            # Use default values
            impact_features = {
                'has_impact': False,
                'impact_count': 0,
                'max_deceleration': 0.0,
                'impact_frame': 0,
                'impact_body_part': None
            }
        
        try:
            collapse_features = analyze_pose_collapse(processed_sequence, fps)
        except Exception as e:
            print(f"Error calculating collapse features: {e}")
            # Use default values
            collapse_features = {
                'height_trajectory': [],
                'width_trajectory': [],
                'aspect_ratio_trajectory': [],
                'height_reduction_pct': 0.0,
                'aspect_ratio_increase_pct': 0.0,
                'max_height_collapse_velocity': 0.0,
                'max_aspect_ratio_change_velocity': 0.0
            }
        
        # Aggregate features over time for key joints
        joint_angle_features = {}
        for joint in ['right_elbow', 'left_elbow', 'right_knee', 'left_knee', 'right_hip', 'left_hip']:
            # Extract angle values over time (with defaults for missing values)
            angle_values = [frame_angles.get(joint, np.nan) for frame_angles in joint_angles_sequence]
            
            # Skip if no valid values, but provide defaults
            if not any(not np.isnan(val) for val in angle_values):
                joint_angle_features[f"{joint}_mean"] = 0.0
                joint_angle_features[f"{joint}_min"] = 0.0
                joint_angle_features[f"{joint}_max"] = 0.0
                joint_angle_features[f"{joint}_range"] = 0.0
                joint_angle_features[f"{joint}_max_velocity"] = 0.0
                continue
            
            # Calculate statistics
            valid_angles = [val for val in angle_values if not np.isnan(val)]
            
            if valid_angles:
                joint_angle_features[f"{joint}_mean"] = np.mean(valid_angles)
                joint_angle_features[f"{joint}_min"] = np.min(valid_angles)
                joint_angle_features[f"{joint}_max"] = np.max(valid_angles)
                joint_angle_features[f"{joint}_range"] = np.max(valid_angles) - np.min(valid_angles)
                
                # Calculate angle velocities
                angle_velocities = calculate_velocity(angle_values, fps)
                valid_velocities = [val for val in angle_velocities if not np.isnan(val)]
                
                if valid_velocities:
                    joint_angle_features[f"{joint}_max_velocity"] = np.max(np.abs(valid_velocities))
                else:
                    joint_angle_features[f"{joint}_max_velocity"] = 0.0
            else:
                # Provide defaults
                joint_angle_features[f"{joint}_mean"] = 0.0
                joint_angle_features[f"{joint}_min"] = 0.0
                joint_angle_features[f"{joint}_max"] = 0.0
                joint_angle_features[f"{joint}_range"] = 0.0
                joint_angle_features[f"{joint}_max_velocity"] = 0.0
        
        # Create features with guaranteed values for all fields
        features = {
            'metadata': {
                'video_path': video_path,
                'fps': fps,
                'total_frames': total_frames,
                'duration': total_frames / fps if fps > 0 else 0,
                'label': label,
                'data_type': 'video'
            },
            'joint_angles': joint_angle_features,
            'vertical_motion': {
                'max_upper_body_velocity_y': vertical_features.get('max_upper_body_velocity_y', 0.0),
                'max_upper_body_accel_y': vertical_features.get('max_upper_body_accel_y', 0.0),
                'max_center_velocity_y': vertical_features.get('max_center_velocity_y', 0.0),
                'max_center_accel_y': vertical_features.get('max_center_accel_y', 0.0),
                'upper_body_vertical_displacement': vertical_features.get('upper_body_vertical_displacement', 0.0),
                'center_vertical_displacement': vertical_features.get('center_vertical_displacement', 0.0),
            },
            'impact': {
                'has_impact': impact_features.get('has_impact', False),
                'impact_count': impact_features.get('impact_count', 0),
                'max_deceleration': impact_features.get('max_deceleration', 0.0),
                'impact_frame': impact_features.get('impact_frame', 0),
                'impact_body_part': impact_features.get('impact_body_part', None),
            },
            'collapse': {
                'height_reduction_pct': collapse_features.get('height_reduction_pct', 0.0),
                'aspect_ratio_increase_pct': collapse_features.get('aspect_ratio_increase_pct', 0.0),
                'max_height_collapse_velocity': collapse_features.get('max_height_collapse_velocity', 0.0),
                'max_aspect_ratio_change_velocity': collapse_features.get('max_aspect_ratio_change_velocity', 0.0),
            }
        }
        
        return features
        
    except Exception as e:
        print(f"Error in feature extraction: {e}")
        return Noneimport os
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# Set up paths
ROOT_DIR = Path(".")
POSE_DATA_DIR = ROOT_DIR / "outputs" / "pose_data"
FEATURES_DIR = ROOT_DIR / "outputs" / "features"
os.makedirs(FEATURES_DIR, exist_ok=True)

# Helper function to load pose data
def load_pose_data(dataset_type):
    """Load all pose data JSON files for a given dataset type"""
    dataset_dir = POSE_DATA_DIR / dataset_type
    if not dataset_dir.exists():
        print(f"Warning: Directory {dataset_dir} does not exist!")
        return []
    
    pose_files = list(dataset_dir.rglob("*.json"))
    print(f"Found {len(pose_files)} pose data files in {dataset_dir}")
    
    pose_data_list = []
    for file_path in tqdm(pose_files, desc=f"Loading {dataset_type} pose data"):
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
                # Add label to the data
                data['label'] = dataset_type
                pose_data_list.append(data)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return pose_data_list

# Helper function to identify data type (video or image)
def identify_data_type(pose_data):
    """Determine whether the pose data is from a video or an image"""
    if 'frames' in pose_data:
        return 'video'
    elif 'people' in pose_data and 'image_path' in pose_data:
        return 'image'
    else:
        return 'unknown'

# 1. Basic Feature Extraction Functions

def calculate_joint_angles(keypoints):
    """
    Calculate joint angles from keypoints - with improved error handling
    
    Args:
        keypoints: List of keypoints [x, y, confidence]
    
    Returns:
        Dictionary of joint angles in degrees
    """
    # Define joint triplets for angle calculation (joint, center, joint)
    joint_triplets = {
        'right_elbow': (5, 7, 9),    # right shoulder, elbow, wrist
        'left_elbow': (6, 8, 10),    # left shoulder, elbow, wrist
        'right_shoulder': (3, 5, 7),  # right ear, shoulder, elbow
        'left_shoulder': (4, 6, 8),   # left ear, shoulder, elbow
        'right_hip': (5, 11, 13),     # right shoulder, hip, knee
        'left_hip': (6, 12, 14),      # left shoulder, hip, knee
        'right_knee': (11, 13, 15),   # right hip, knee, ankle
        'left_knee': (12, 14, 16),    # left hip, knee, ankle
        'neck': (5, 0, 6)             # right shoulder, nose, left shoulder
    }
    
    angles = {}
    
    # Check if keypoints is a valid list
    if not isinstance(keypoints, list) or len(keypoints) < 17:
        print(f"Invalid keypoints data: received {type(keypoints)} with length {len(keypoints) if isinstance(keypoints, list) else 'N/A'}")
        # Return default values instead of empty dict
        for joint_name in joint_triplets:
            angles[joint_name] = 0.0
        return angles
    
    for joint_name, (p1_idx, p2_idx, p3_idx) in joint_triplets.items():
        # Check if indices are valid
        if max(p1_idx, p2_idx, p3_idx) >= len(keypoints):
            angles[joint_name] = 0.0  # Use 0 instead of NaN
            continue
            
        # Check if keypoints have the right format
        try:
            # Skip if any keypoint has low confidence
            if (len(keypoints[p1_idx]) < 3 or len(keypoints[p2_idx]) < 3 or len(keypoints[p3_idx]) < 3 or
                keypoints[p1_idx][2] < 0.5 or keypoints[p2_idx][2] < 0.5 or keypoints[p3_idx][2] < 0.5):
                angles[joint_name] = 0.0  # Use 0 instead of NaN
                continue
            
            # Get coordinates
            p1 = keypoints[p1_idx][:2]  # First point
            p2 = keypoints[p2_idx][:2]  # Center point (the joint)
            p3 = keypoints[p3_idx][:2]  # Third point
            
            # Calculate vectors
            v1 = np.array([p1[0] - p2[0], p1[1] - p2[1]])
            v2 = np.array([p3[0] - p2[0], p3[1] - p2[1]])
            
            # Normalize vectors
            v1_norm = np.linalg.norm(v1)
            v2_norm = np.linalg.norm(v2)
            
            if v1_norm == 0 or v2_norm == 0:
                angles[joint_name] = 0.0  # Use 0 instead of NaN
                continue
            
            v1 = v1 / v1_norm
            v2 = v2 / v2_norm
            
            # Calculate dot product and angle
            dot_product = np.clip(np.dot(v1, v2), -1.0, 1.0)
            angle = np.arccos(dot_product) * 180 / np.pi
            
            angles[joint_name] = angle
            
        except Exception as e:
            print(f"Error calculating {joint_name} angle: {str(e)}")
            angles[joint_name] = 0.0  # Use 0 instead of NaN
    
    return angles

def calculate_body_proportions(keypoints):
    """
    Calculate body proportions (ratios of body part lengths)
    
    Args:
        keypoints: List of keypoints [x, y, confidence]
    
    Returns:
        Dictionary of body proportions
    """
    # Define segments for calculation
    segments = {
        'torso': (5, 11),  # right shoulder to right hip
        'right_arm': (5, 9),  # right shoulder to right wrist
        'left_arm': (6, 10),  # left shoulder to left wrist
        'right_leg': (11, 15),  # right hip to right ankle
        'left_leg': (12, 16),  # left hip to left ankle
        'shoulders': (5, 6),  # right shoulder to left shoulder
        'hips': (11, 12)  # right hip to left hip
    }
    
    # Calculate segment lengths
    lengths = {}
    for segment_name, (p1_idx, p2_idx) in segments.items():
        # Skip if any keypoint has low confidence
        if (p1_idx >= len(keypoints) or p2_idx >= len(keypoints) or
            len(keypoints[p1_idx]) < 3 or len(keypoints[p2_idx]) < 3 or
            keypoints[p1_idx][2] < 0.5 or keypoints[p2_idx][2] < 0.5):
            lengths[segment_name] = 0.0  # Use 0 instead of NaN
            continue
        
        # Get coordinates
        p1 = keypoints[p1_idx][:2]
        p2 = keypoints[p2_idx][:2]
        
        # Calculate Euclidean distance
        length = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
        lengths[segment_name] = length
    
    # Calculate proportions
    proportions = {}
    
    # Initialize all proportions with defaults
    for segment_name in segments:
        if segment_name != 'torso':
            proportions[f"{segment_name}_ratio"] = 1.0  # Default ratio value
    
    proportions['shoulder_hip_ratio'] = 1.0
    proportions['width_height_ratio'] = 1.0
    
    # Calculate actual proportions where possible
    if 'torso' in lengths and lengths['torso'] > 0:
        for segment_name, length in lengths.items():
            if segment_name != 'torso' and length > 0:
                proportions[f"{segment_name}_ratio"] = length / lengths['torso']
    
    # Add shoulder-to-hip ratio (width vs height)
    if ('shoulders' in lengths and 'hips' in lengths and 'torso' in lengths and
        lengths['shoulders'] > 0 and lengths['hips'] > 0 and lengths['torso'] > 0):
        proportions['shoulder_hip_ratio'] = lengths['shoulders'] / lengths['hips']
        proportions['width_height_ratio'] = lengths['shoulders'] / lengths['torso']
    
    return proportions

def calculate_pose_center(keypoints):
    """
    Calculate the center of the pose
    
    Args:
        keypoints: List of keypoints [x, y, confidence]
    
    Returns:
        Center coordinates [x, y]
    """
    # Consider only high-confidence keypoints
    valid_points = []
    for kp in keypoints:
        if len(kp) >= 3 and kp[2] >= 0.5:
            valid_points.append(kp[:2])
    
    if not valid_points:
        return [0.0, 0.0]  # Use [0,0] instead of NaN
    
    # Calculate mean position
    valid_points = np.array(valid_points)
    center = np.mean(valid_points, axis=0)
    
    return center.tolist()

def calculate_bounding_box(keypoints):
    """
    Calculate the bounding box of the pose
    
    Args:
        keypoints: List of keypoints [x, y, confidence]
    
    Returns:
        Dictionary with bounding box properties: x, y, width, height, aspect_ratio
    """
    # Filter valid keypoints
    valid_x = []
    valid_y = []
    
    for kp in keypoints:
        if len(kp) >= 3 and kp[2] >= 0.5:
            valid_x.append(kp[0])
            valid_y.append(kp[1])
    
    if not valid_x or not valid_y:
        return {
            'x': 0.0, 'y': 0.0, 
            'width': 0.0, 'height': 0.0, 
            'area': 0.0, 'aspect_ratio': 1.0  # Default aspect ratio
        }
    
    # Calculate bounding box
    min_x = min(valid_x)
    max_x = max(valid_x)
    min_y = min(valid_y)
    max_y = max(valid_y)
    
    width = max_x - min_x
    height = max_y - min_y
    area = width * height
    aspect_ratio = width / height if height > 0 else 1.0  # Default to 1.0 if height is 0
    
    return {
        'x': min_x,
        'y': min_y,
        'width': width,
        'height': height,
        'area': area,
        'aspect_ratio': aspect_ratio
    }

# 2. Temporal Feature Extraction Functions with fixes

def smooth_signal(data, window_size=5):
    """
    Apply smoothing to a signal
    
    Args:
        data: Time series data
        window_size: Size of the moving average window
    
    Returns:
        Smoothed data
    """
    if len(data) < window_size:
        return data
    
    # Convert to numpy array for easier processing
    data = np.array(data)
    
    # Handle NaN values
    nan_mask = np.isnan(data)
    
    # If all values are NaN, return same array
    if np.all(nan_mask):
        return data.tolist()
    
    # Create a copy for processing
    data_cleaned = data.copy()
    
    # Fill NaN values with interpolation where possible
    if np.any(nan_mask):
        # Get indices of valid values
        valid_indices = np.where(~nan_mask)[0]
        
        # For each invalid value
        for i in np.where(nan_mask)[0]:
            # Find nearest valid values before and after
            prev_valid_idx = valid_indices[valid_indices < i]
            next_valid_idx = valid_indices[valid_indices > i]
            
            if len(prev_valid_idx) > 0 and len(next_valid_idx) > 0:
                # Interpolate between nearest valid values
                prev_idx = prev_valid_idx[-1]
                next_idx = next_valid_idx[0]
                weight = (i - prev_idx) / (next_idx - prev_idx)
                data_cleaned[i] = data[prev_idx] + weight * (data[next_idx] - data[prev_idx])
            elif len(prev_valid_idx) > 0:
                # Use last valid value
                data_cleaned[i] = data[prev_valid_idx[-1]]
            elif len(next_valid_idx) > 0:
                # Use next valid value
                data_cleaned[i] = data[next_valid_idx[0]]
    
    # Create a moving average kernel
    kernel = np.ones(window_size) / window_size
    
    # Apply convolution for smoothing
    smoothed_data = np.convolve(data_cleaned, kernel, mode='same')
    
    # Handle edge effects
    half_window = window_size // 2
    smoothed_data[:half_window] = data_cleaned[:half_window]
    smoothed_data[-half_window:] = data_cleaned[-half_window:]
    
    return smoothed_data.tolist()

def calculate_velocity(positions, fps):
    """
    Improved velocity calculation that handles missing data
    
    Args:
        positions: List of position values over time
        fps: Frames per second
    
    Returns:
        List of velocities
    """
    if len(positions) < 2:
        return []
    
    # Convert to numpy array for easier processing
    positions = np.array(positions)
    
    # Create a mask for valid values (not NaN)
    valid_mask = ~np.isnan(positions)
    
    # If all values are NaN, return zeros
    if not np.any(valid_mask):
        return np.zeros(len(positions)).tolist()
    
    # Create a copy for processing
    positions_cleaned = positions.copy()
    
    # Fill NaN values with interpolation where possible
    if np.any(~valid_mask):
        # Get indices of valid values
        valid_indices = np.where(valid_mask)[0]
        
        # For each invalid value
        for i in np.where(~valid_mask)[0]:
            # Find nearest valid values before and after
            prev_valid_idx = valid_indices[valid_indices < i]
            next_valid_idx = valid_indices[valid_indices > i]
            
            if len(prev_valid_idx) > 0 and len(next_valid_idx) > 0:
                # Interpolate between nearest valid values
                prev_idx = prev_valid_idx[-1]
                next_idx = next_valid_idx[0]
                weight = (i - prev_idx) / (next_idx - prev_idx)
                positions_cleaned[i] = positions[prev_idx] + weight * (positions[next_idx] - positions[prev_idx])
            elif len(prev_valid_idx) > 0:
                # Use last valid value
                positions_cleaned[i] = positions[prev_valid_idx[-1]]
            elif len(next_valid_idx) > 0:
                # Use next valid value
                positions_cleaned[i] = positions[next_valid_idx[0]]
    
    # Calculate velocity (change in position per second)
    velocities = np.diff(positions_cleaned) * fps
    
    # Add 0 as first velocity for consistent length
    return np.concatenate(([0], velocities)).tolist()

def calculate_acceleration(velocities, fps):
    """
    Improved acceleration calculation that handles missing data
    
    Args:
        velocities: List of velocity values over time
        fps: Frames per second
    
    Returns:
        List of accelerations
    """
    if len(velocities) < 2:
        return []
    
    # Convert to numpy array for easier processing
    velocities = np.array(velocities)
    
    # Create a mask for valid values (not NaN)
    valid_mask = ~np.isnan(velocities)
    
    # If all values are NaN, return zeros
    if not np.any(valid_mask):
        return np.zeros(len(velocities)).tolist()
    
    # Create a copy for processing
    velocities_cleaned = velocities.copy()
    
    # Fill NaN values with interpolation where possible
    if np.any(~valid_mask):
        # Get indices of valid values
        valid_indices = np.where(valid_mask)[0]
        
        # For each invalid value
        for i in np.where(~valid_mask)[0]:
            # Find nearest valid values before and after
            prev_valid_idx = valid_indices[valid_indices < i]
            next_valid_idx = valid_indices[valid_indices > i]
            
            if len(prev_valid_idx) > 0 and len(next_valid_idx) > 0:
                # Interpolate between nearest valid values
                prev_idx = prev_valid_idx[-1]
                next_idx = next_valid_idx[0]
                weight = (i - prev_idx) / (next_idx - prev_idx)
                velocities_cleaned[i] = velocities[prev_idx] + weight * (velocities[next_idx] - velocities[prev_idx])
            elif len(prev_valid_idx) > 0:
                # Use last valid value
                velocities_cleaned[i] = velocities[prev_valid_idx[-1]]
            elif len(next_valid_idx) > 0:
                # Use next valid value
                velocities_cleaned[i] = velocities[next_valid_idx[0]]
    
    # Calculate acceleration (change in velocity per second)
    accelerations = np.diff(velocities_cleaned) * fps
    
    # Add 0 as first acceleration for consistent length
    return np.concatenate(([0], accelerations)).tolist()

# 3. Fall-Specific Feature Functions with improved error handling

def calculate_vertical_displacement(keypoints_sequence, fps):
    """
    Calculate vertical displacement features for fall detection
    
    Args:
        keypoints_sequence: List of keypoints over time
        fps: Frames per second
    
    Returns:
        Dictionary of vertical displacement features
    """
    # Track center of upper body (nose, shoulders) over time
    upper_body_y = []
    center_y = []
    
    for frame_keypoints in keypoints_sequence:
        # Upper body markers (nose, shoulders)
        upper_markers = [0, 5, 6]  # Indices for nose, right shoulder, left shoulder
        
        # Calculate average y-position of upper body
        valid_y = []
        for idx in upper_markers:
            if idx < len(frame_keypoints) and (len(frame_keypoints[idx]) >= 3 and 
                frame_keypoints[idx][2] >= 0.5):
                valid_y.append(frame_keypoints[idx][1])
        
        if valid_y:
            upper_body_y.append(np.mean(valid_y))
        else:
            upper_body_y.append(np.nan)
        
        # Calculate overall center y-position
        center = calculate_pose_center(frame_keypoints)
        center_y.append(center[1])
    
    # Smooth the trajectories to reduce noise
    upper_body_y = smooth_signal(upper_body_y, window_size=min(5, len(upper_body_y)))
    center_y = smooth_signal(center_y, window_size=min(5, len(center_y)))
    
    # Calculate velocity and acceleration
    upper_body_velocity_y = calculate_velocity(upper_body_y, fps)
    upper_body_accel_y = calculate_acceleration(upper_body_velocity_y, fps)
    
    center_velocity_y = calculate_velocity(center_y, fps)
    center_accel_y = calculate_acceleration(center_velocity_y, fps)
    
    # Calculate maximum values and rates of change
    max_upper_vel = np.nanmax(np.abs(upper_body_velocity_y)) if upper_body_velocity_y and not np.all(np.isnan(upper_body_velocity_y)) else 0.0
    max_upper_acc = np.nanmax(np.abs(upper_body_accel_y)) if upper_body_accel_y and not np.all(np.isnan(upper_body_accel_y)) else 0.0
    
    max_center_vel = np.nanmax(np.abs(center_velocity_y)) if center_velocity_y and not np.all(np.isnan(center_velocity_y)) else 0.0
    max_center_acc = np.nanmax(np.abs(center_accel_y)) if center_accel_y and not np.all(np.isnan(center_accel_y)) else 0.0
    
    # Calculate total vertical displacement (first to last frame)
    if len(upper_body_y) > 1:
        # Find first and last valid values
        valid_indices = [i for i, y in enumerate(upper_body_y) if not np.isnan(y)]
        if valid_indices:
            first_idx = valid_indices[0]
            last_idx = valid_indices[-1]
            upper_body_displacement = upper_body_y[last_idx] - upper_body_y[first_idx]
        else:
            upper_body_displacement = 0.0
    else:
        upper_body_displacement = 0.0
    
    if len(center_y) > 1:
        # Find first and last valid values
        valid_indices = [i for i, y in enumerate(center_y) if not np.isnan(y)]
        if valid_indices:
            first_idx = valid_indices[0]
            last_idx = valid_indices[-1]
            center_displacement = center_y[last_idx] - center_y[first_idx]
        else:
            center_displacement = 0.0
    else:
        center_displacement = 0.0
    
    return {
        'upper_body_y_trajectory': upper_body_y,
        'center_y_trajectory': center_y,
        'upper_body_velocity_y': upper_body_velocity_y,
        'upper_body_accel_y': upper_body_accel_y,
        'center_velocity_y': center_velocity_y,
        'center_accel_y': center_accel_y,
        'max_upper_body_velocity_y': max_upper_vel,
        'max_upper_body_accel_y': max_upper_acc,
        'max_center_velocity_y': max_center_vel,
        'max_center_accel_y': max_center_acc,
        'upper_body_vertical_displacement': upper_body_displacement,
        'center_vertical_displacement': center_displacement
    }

def detect_impact(keypoints_sequence, fps, threshold=2.0):
    """
    Detect potential impact events in the sequence
    
    Args:
        keypoints_sequence: List of keypoints over time
        fps: Frames per second
        threshold: Acceleration threshold (in units/s²) for impact detection
    
    Returns:
        Dictionary of impact features
    """
    # Initialize default return for safety
    default_impact = {
        'has_impact': False,
        'impact_count': 0,
        'max_deceleration': 0.0,
        'impact_frame': 0,
        'impact_body_part': None
    }
    
    # Skip if sequence is too short
    if len(keypoints_sequence) < 3:
        return default_impact
    
    # Track body parts likely to impact during a fall
    tracked_keypoints = {
        'head': 0,  # nose
        'hip_center': None,  # will calculate as average of left and right hip
        'knee_center': None,  # will calculate as average of left and right knee
    }
    
    # Track positions over time
    trajectories = {part: {'x': [], 'y': []} for part in tracked_keypoints}
    
    for frame_keypoints in keypoints_sequence:
        # Process single keypoints
        for part, idx in tracked_keypoints.items():
            if idx is not None:
                if idx < len(frame_keypoints) and (len(frame_keypoints[idx]) >= 3 and 
                    frame_keypoints[idx][2] >= 0.5):
                    trajectories[part]['x'].append(frame_keypoints[idx][0])
                    trajectories[part]['y'].append(frame_keypoints[idx][1])
                else:
                    trajectories[part]['x'].append(np.nan)
                    trajectories[part]['y'].append(np.nan)
        
        # Calculate hip center
        left_hip_idx, right_hip_idx = 12, 11  # left hip, right hip
        if (left_hip_idx < len(frame_keypoints) and right_hip_idx < len(frame_keypoints) and
            len(frame_keypoints[left_hip_idx]) >= 3 and len(frame_keypoints[right_hip_idx]) >= 3 and
            frame_keypoints[left_hip_idx][2] >= 0.5 and frame_keypoints[right_hip_idx][2] >= 0.5):
            hip_center_x = (frame_keypoints[left_hip_idx][0] + frame_keypoints[right_hip_idx][0]) / 2
            hip_center_y = (frame_keypoints[left_hip_idx][1] + frame_keypoints[right_hip_idx][1]) / 2
            trajectories['hip_center']['x'].append(hip_center_x)
            trajectories['hip_center']['y'].append(hip_center_y)
        else:
            trajectories['hip_center']['x'].append(np.nan)
            trajectories['hip_center']['y'].append(np.nan)
        
        # Calculate knee center
        left_knee_idx, right_knee_idx = 14, 13  # left knee, right knee
        if (left_knee_idx < len(frame_keypoints) and right_knee_idx < len(frame_keypoints) and
            len(frame_keypoints[left_knee_idx]) >= 3 and len(frame_keypoints[right_knee_idx]) >= 3 and
            frame_keypoints[left_knee_idx][2] >= 0.5 and frame_keypoints[right_knee_idx][2] >= 0.5):
            knee_center_x = (frame_keypoints[left_knee_idx][0] + frame_keypoints[right_knee_idx][0]) / 2
            knee_center_y = (frame_keypoints[left_knee_idx][1] + frame_keypoints[right_knee_idx][1]) / 2
            trajectories['knee_center']['x'].append(knee_center_x)
            trajectories['knee_center']['y'].append(knee_center_y)
        else:
            trajectories['knee_center']['x'].append(np.nan)
            trajectories['knee_center']['y'].append(np.nan)
    
    # Check if we have any valid trajectory data
    has_valid_data = False
    for part in trajectories:
        for coord in ['x', 'y']:
            # Check if there's at least one valid (non-NaN) value
            if any(not np.isnan(val) for val in trajectories[part][coord]):
                has_valid_data = True
                break
        if has_valid_data:
            break
    
    # Return default if no valid data
    if not has_valid_data:
        return default_impact
    
    # Smooth trajectories
    for part in trajectories:
        for coord in ['x', 'y']:
            trajectories[part][coord] = smooth_signal(
                trajectories[part][coord], 
                window_size=min(5, len(trajectories[part][coord]))
            )
    
    # Calculate velocities and accelerations
    velocities = {part: {'x': None, 'y': None} for part in tracked_keypoints}
    accelerations = {part: {'x': None, 'y': None} for part in tracked_keypoints}
    
    for part in trajectories:
        for coord in ['x', 'y']:
            velocities[part][coord] = calculate_velocity(trajectories[part][coord], fps)
            accelerations[part][coord] = calculate_acceleration(velocities[part][coord], fps)
    
    # Detect impact events (rapid deceleration)
    impact_events = {}
    try:
        for part in accelerations:
            for coord in ['x', 'y']:
                # Skip if acceleration data is empty or all NaN
                if not accelerations[part][coord] or np.all(np.isnan(accelerations[part][coord])):
                    continue
                
                # Convert to numpy array and handle NaN values
                acc_data = np.array(accelerations[part][coord])
                acc_data = np.nan_to_num(acc_data, nan=0.0)
                
                # Find minimum acceleration (maximum deceleration)
                min_acc_idx = np.argmin(acc_data)
                min_acc_val = acc_data[min_acc_idx]
                
                # If there's a significant deceleration
                if min_acc_val < -threshold:
                    # Get velocity before impact
                    if min_acc_idx > 0 and min_acc_idx < len(velocities[part][coord]):
                        vel_before = velocities[part][coord][min_acc_idx - 1]
                    else:
                        vel_before = 0.0
                    
                    impact_events[f"{part}_{coord}"] = {
                        'frame_idx': int(min_acc_idx),
                        'deceleration': float(min_acc_val),
                        'velocity_before': float(vel_before)
                    }
        
        # Calculate summary features
        if impact_events:
            # Find the strongest impact
            max_decel = min([event['deceleration'] for event in impact_events.values()])
            max_part = next(key for key, val in impact_events.items() if val['deceleration'] == max_decel)
            
            return {
                'has_impact': True,
                'impact_count': len(impact_events),
                'max_deceleration': float(impact_events[max_part]['deceleration']),
                'impact_frame': int(impact_events[max_part]['frame_idx']),
                'impact_body_part': max_part
            }
        else:
            return default_impact
            
    except Exception as e:
        print(f"Error in impact detection: {e}")
        return default_impact

def analyze_pose_collapse(keypoints_sequence, fps):
    """
    Analyze pose collapse patterns (typical in falls)
    
    Args:
        keypoints_sequence: List of keypoints over time
        fps: Frames per second
    
    Returns:
        Dictionary of pose collapse features
    """
    # Initialize default return for safety
    default_collapse = {
        'height_trajectory': [],
        'width_trajectory': [],
        'aspect_ratio_trajectory': [],
        'height_reduction_pct': 0.0,
        'aspect_ratio_increase_pct': 0.0,
        'max_height_collapse_velocity': 0.0,
        'max_aspect_ratio_change_velocity': 0.0
    }
    
    # Skip if sequence is too short
    if len(keypoints_sequence) < 3:
        return default_collapse
    
    # Track body proportions over time
    height_values = []  # Vertical height
    width_values = []   # Horizontal width
    aspect_ratios = []  # Width/height ratio
    
    for frame_keypoints in keypoints_sequence:
        # Calculate bounding box
        bbox = calculate_bounding_box(frame_keypoints)
        
        height_values.append(bbox['height'])
        width_values.append(bbox['width'])
        aspect_ratios.append(bbox['aspect_ratio'])
    
    # Replace NaN values with interpolation or nearest valid values
    height_values = smooth_signal(height_values, window_size=min(5, len(height_values)))
    width_values = smooth_signal(width_values, window_size=min(5, len(width_values)))
    aspect_ratios = smooth_signal(aspect_ratios, window_size=min(5, len(aspect_ratios)))
    
    # Calculate relative changes
    if len(height_values) > 1:
        # Find valid values for first frame
        valid_height_indices = [i for i, h in enumerate(height_values) if h > 0]
        valid_ar_indices = [i for i, ar in enumerate(aspect_ratios) if ar > 0 and not np.isnan(ar)]
        
        if valid_height_indices and valid_height_indices[0] < len(height_values) - 1:
            # Use first valid height as reference
            first_height_idx = valid_height_indices[0]
            first_height = height_values[first_height_idx]
            
            # Find minimum height after first valid frame
            later_heights = height_values[first_height_idx+1:]
            if later_heights:
                min_height = min([h for h in later_heights if h > 0], default=first_height)
                
                # Calculate percentage reduction
                if first_height > 0:
                    height_reduction = (first_height - min_height) / first_height
                else:
                    height_reduction = 0.0
            else:
                height_reduction = 0.0
        else:
            height_reduction = 0.0
        
        # Calculate aspect ratio increase similarly
        if valid_ar_indices and valid_ar_indices[0] < len(aspect_ratios) - 1:
            # Use first valid aspect ratio as reference
            first_ar_idx = valid_ar_indices[0]
            first_ar = aspect_ratios[first_ar_idx]
            
            # Find maximum aspect ratio after first valid frame
            later_ars = aspect_ratios[first_ar_idx+1:]
            if later_ars:
                max_ar = max([ar for ar in later_ars if not np.isnan(ar)], default=first_ar)
                
                # Calculate percentage increase
                if first_ar > 0:
                    ar_increase = (max_ar - first_ar) / first_ar
                else:
                    ar_increase = 0.0
            else:
                ar_increase = 0.0
        else:
            ar_increase = 0.0
    else:
        height_reduction = 0.0
        ar_increase = 0.0
    
    # Calculate velocities for changes
    height_velocity = calculate_velocity(height_values, fps)
    aspect_ratio_velocity = calculate_velocity(aspect_ratios, fps)
    
    # Calculate maximum rate of change
    if height_velocity and not np.all(np.isnan(height_velocity)):
        # Find most negative velocity (fastest collapse)
        valid_height_vel = [v for v in height_velocity if not np.isnan(v)]
        max_height_velocity = min(valid_height_vel, default=0.0)
    else:
        max_height_velocity = 0.0
    
    if aspect_ratio_velocity and not np.all(np.isnan(aspect_ratio_velocity)):
        # Find most positive velocity (fastest widening)
        valid_ar_vel = [v for v in aspect_ratio_velocity if not np.isnan(v)]
        max_ar_velocity = max(valid_ar_vel, default=0.0)
    else:
        max_ar_velocity = 0.0
    
    return {
        'height_trajectory': height_values,
        'width_trajectory': width_values,
        'aspect_ratio_trajectory': aspect_ratios,
        'height_reduction_pct': height_reduction * 100,  # Convert to percentage
        'aspect_ratio_increase_pct': ar_increase * 100,  # Convert to percentage
        'max_height_collapse_velocity': max_height_velocity,
        'max_aspect_ratio_change_velocity': max_ar_velocity
    }

SyntaxError: invalid syntax (4226314697.py, line 971)